In [8]:
# ============================================================
# LANGKAH 1: Import Library & Load Dataset
# ============================================================
import pandas as pd
import numpy as np

# 1. Load Dataset
df = pd.read_csv('../dialogue/output/tabel_perbandingan_lengkap.csv')
print(f"Total data: {len(df)}")
print(f"Kolom: {list(df.columns)}")
print(f"\nLabel unik: {df['label'].unique()}")
print(f"Jenis unik: {df['jenis'].unique()}")
print(f"\nJumlah per label & jenis:")
print(df.groupby(['label', 'jenis']).size())
print("\n5 data pertama:")
df.head()

Total data: 42
Kolom: ['label', 'jenis', 'teks_dialog', 'teks_kritik', 'skor_critic']

Label unik: <StringArray>
['good', 'bad']
Length: 2, dtype: str
Jenis unik: <StringArray>
['original', 'rephrase']
Length: 2, dtype: str

Jumlah per label & jenis:
label  jenis   
bad    original     1
       rephrase    20
good   original     1
       rephrase    20
dtype: int64

5 data pertama:


,label,jenis,teks_dialog,teks_kritik,skor_critic
0,good,original,"Eh kak, akhirnya sampai juga! Pesen Green Tea ...",Dialog jelas disampaikan oleh Raka saat memesa...,4.25
1,good,rephrase,"Eh kak, akhirnya tiba juga! Aku pesan Green Te...",Dialog relevan dengan state pesanan karena Rak...,4.50
2,good,rephrase,"Eh kak, akhirnya tiba juga! Aku pesan Green Te...",Dialog jelas disampaikan oleh Raka saat memesa...,4.50
3,good,rephrase,"Eh kak, akhirnya tiba juga! Aku pesan Green Te...",Dialog menyampaikan pesanan Green Tea secara j...,4.75
4,good,rephrase,"Eh kak, akhirnya tiba juga! Aku pesan Green Te...",Dialog menarik dan spesifik karena Raka menjel...,4.50


In [9]:
# ============================================================
# LANGKAH 2: Load Model Sentence Embedding (HuggingFace)
# ============================================================
# Model: paraphrase-multilingual-MiniLM-L12-v2
# Model ini menghasilkan vektor embedding 384 dimensi yang
# merepresentasikan makna semantik dari teks (termasuk Bahasa Indonesia)
import torch
from transformers import AutoTokenizer, AutoModel

model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"Model '{model_name}' berhasil dimuat!")
print(f"Dimensi embedding: {model.config.hidden_size}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' berhasil dimuat!
Dimensi embedding: 384


In [10]:
# ============================================================
# LANGKAH 3: Fungsi untuk Membuat Sentence Embedding
# ============================================================
# Teknik: Mean Pooling — mengambil rata-rata vektor dari
# seluruh hidden states untuk menghasilkan satu vektor
# representasi semantik dari teks input
def get_embedding(texts):
    """
    Mengubah teks menjadi vektor embedding numerik.
    Input: list of strings
    Output: numpy array (n_texts, 384)
    """
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling — rata-rata vektor dari hidden states
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

# Contoh: embedding dari teks_kritik original (data asli dari dataset)
contoh_teks = df[df['jenis'] == 'original']['teks_kritik'].iloc[0]
contoh_embedding = get_embedding([contoh_teks])
print(f"Contoh embedding untuk teks_kritik original:")
print(f"  '{contoh_teks[:100]}...'")
print(f"  Shape: {contoh_embedding.shape}")
print(f"  5 nilai pertama: {contoh_embedding[0][:5]}")

Contoh embedding untuk teks_kritik original:
  'Dialog jelas disampaikan oleh Raka saat memesan Green Tea dan relevan dengan state pesanan. Detail d...'
  Shape: (1, 384)
  5 nilai pertama: [ 0.16348118  0.1222467  -0.16517171  0.04107502 -0.02380887]


In [11]:
# ============================================================
# LANGKAH 4: Hitung Cosine Similarity — Original vs Rephrase
# ============================================================
# Alur:
# 1. Ambil teks_kritik ORIGINAL (jenis='original') sbg acuan
# 2. Ambil semua teks_kritik REPHRASE (jenis='rephrase')
# 3. Generate embedding untuk original dan semua rephrase
# 4. Hitung cosine similarity antara embedding original dan
#    masing-masing embedding rephrase
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(dataframe):
    results_list = []
    summary_list = []
    
    for label_name in ['good', 'bad']:
        sub_df = dataframe[dataframe['label'] == label_name].copy()
        
        # Ambil teks kritik ORIGINAL sebagai acuan
        orig_row = sub_df[sub_df['jenis'] == 'original'].iloc[0]
        orig_text = orig_row['teks_kritik']
        
        # Ambil semua teks kritik REPHRASE
        rephrase_df = sub_df[sub_df['jenis'] == 'rephrase'].copy()
        rephrase_texts = rephrase_df['teks_kritik'].tolist()
        
        # Generate embedding
        orig_embedding = get_embedding([orig_text])
        rephrase_embeddings = get_embedding(rephrase_texts)
        
        # Hitung cosine similarity
        sim_scores = cosine_similarity(orig_embedding, rephrase_embeddings).flatten()
        
        # Simpan ke dataframe detail
        rephrase_df['cosine_similarity'] = sim_scores
        results_list.append(rephrase_df)
        
        # Simpan ringkasan per label
        summary_list.append({
            'label': label_name,
            'mean_cosine_similarity': np.mean(sim_scores),
            'std_cosine_similarity': np.std(sim_scores),
            'min_cosine_similarity': np.min(sim_scores),
            'max_cosine_similarity': np.max(sim_scores)
        })
    
    df_detailed = pd.concat(results_list, ignore_index=True)
    df_summary = pd.DataFrame(summary_list)
    
    return df_detailed, df_summary

# Jalankan fungsi
df_detailed, df_summary = calculate_cosine_similarity(df)
print("Perhitungan cosine similarity selesai!")
print(f"Total data hasil: {len(df_detailed)}")

Perhitungan cosine similarity selesai!
Total data hasil: 40


In [12]:
# ============================================================
# LANGKAH 5: Ringkasan Cosine Similarity per Label
# ============================================================
print("=== RINGKASAN COSINE SIMILARITY PER LABEL ===")
df_summary

=== RINGKASAN COSINE SIMILARITY PER LABEL ===


,label,mean_cosine_similarity,std_cosine_similarity,min_cosine_similarity,max_cosine_similarity
0,good,0.889393,0.043757,0.763526,0.945622
1,bad,0.900058,0.029422,0.831083,0.946954


In [13]:
# ============================================================
# LANGKAH 6: Statistik Detail per Label
# ============================================================
print("=== STATISTIK COSINE SIMILARITY PER LABEL ===\n")

# Filter hanya data rephrase
df_rephrase_cosine = df_detailed[['label', 'cosine_similarity']].copy()

for label in ['good', 'bad']:
    label_data = df_rephrase_cosine[df_rephrase_cosine['label'] == label]
    print(f"Label '{label}':")
    print(f"  Jumlah sampel      : {len(label_data)}")
    print(f"  Rata-rata (mean)   : {label_data['cosine_similarity'].mean():.4f}")
    print(f"  Standar deviasi    : {label_data['cosine_similarity'].std():.4f}")
    print(f"  Nilai minimum      : {label_data['cosine_similarity'].min():.4f}")
    print(f"  Nilai maksimum     : {label_data['cosine_similarity'].max():.4f}")
    print()

print("=== RINGKASAN KESELURUHAN ===")
print(f"  Total sampel       : {len(df_rephrase_cosine)}")
print(f"  Rata-rata (mean)   : {df_rephrase_cosine['cosine_similarity'].mean():.4f}")
print(f"  Standar deviasi    : {df_rephrase_cosine['cosine_similarity'].std():.4f}")

=== STATISTIK COSINE SIMILARITY PER LABEL ===

Label 'good':
  Jumlah sampel      : 20
  Rata-rata (mean)   : 0.8894
  Standar deviasi    : 0.0449
  Nilai minimum      : 0.7635
  Nilai maksimum     : 0.9456

Label 'bad':
  Jumlah sampel      : 20
  Rata-rata (mean)   : 0.9001
  Standar deviasi    : 0.0302
  Nilai minimum      : 0.8311
  Nilai maksimum     : 0.9470

=== RINGKASAN KESELURUHAN ===
  Total sampel       : 40
  Rata-rata (mean)   : 0.8947
  Standar deviasi    : 0.0381


In [14]:
# ============================================================
# LANGKAH 7: Tabel Lengkap Cosine Similarity (Semua Data)
# ============================================================
print("=== TABEL COSINE SIMILARITY — SEMUA DATA REPHRASE ===\n")
print(f"Total data: {len(df_rephrase_cosine)}\n")
display(df_rephrase_cosine)

=== TABEL COSINE SIMILARITY — SEMUA DATA REPHRASE ===

Total data: 40



,label,cosine_similarity
0,good,0.888549
1,good,0.887822
2,good,0.838728
3,good,0.763526
4,good,0.944159
5,good,0.830975
6,good,0.945622
7,good,0.928580
8,good,0.936964
9,good,0.932806


In [15]:
# ============================================================
# LANGKAH 8: Simpan Hasil ke CSV
# ============================================================
df_detailed.to_csv('hasil_cosine_similarity.csv', index=False)
print("Hasil detail berhasil disimpan ke 'hasil_cosine_similarity.csv'!")
print(f"Kolom tersimpan: {list(df_detailed.columns)}")

Hasil detail berhasil disimpan ke 'hasil_cosine_similarity.csv'!
Kolom tersimpan: ['label', 'jenis', 'teks_dialog', 'teks_kritik', 'skor_critic', 'cosine_similarity']


In [16]:
# ============================================================
# TABEL ASAL — Good 3 & Bad 3 (Rephrase only)
# ============================================================
import pandas as pd

df_asal = pd.read_csv('../dialogue/output/tabel_perbandingan_lengkap.csv')

df_good = df_asal[(df_asal['label'] == 'good') & (df_asal['jenis'] == 'rephrase')].head(3)
df_bad = df_asal[(df_asal['label'] == 'bad') & (df_asal['jenis'] == 'rephrase')].head(3)
df_tabel = pd.concat([df_good, df_bad], ignore_index=True)

# Drop kolom teks_dialog karena isinya sama semua
df_tabel = df_tabel.drop(columns=['teks_dialog'])

display(df_tabel)

,label,jenis,teks_kritik,skor_critic
0,good,rephrase,Dialog relevan dengan state pesanan karena Rak...,4.500
1,good,rephrase,Dialog jelas disampaikan oleh Raka saat memesa...,4.500
2,good,rephrase,Dialog menyampaikan pesanan Green Tea secara j...,4.750
3,bad,rephrase,Dialog menyampaikan pesanan Green Tea dan kond...,3.500
4,bad,rephrase,Dialog menyampaikan pesanan Green Tea dan kond...,3.500
5,bad,rephrase,Dialog jelas menyampaikan pesanan Green Tea da...,3.625


In [17]:
# ============================================================
# TABEL EMBEDDING VECTOR — Good 3 & Bad 3 (Original + Rephrase)
# ============================================================
import pandas as pd

df_asal = pd.read_csv('../dialogue/output/tabel_perbandingan_lengkap.csv')

# Ambil 3 good + 3 bad (masing-masing: 1 original + 2 rephrase)
df_good = df_asal[df_asal['label'] == 'good'].head(3)
df_bad = df_asal[df_asal['label'] == 'bad'].head(3)
df_sample = pd.concat([df_good, df_bad], ignore_index=True)

# Generate embedding untuk teks_kritik
teks_list = df_sample['teks_kritik'].tolist()
embeddings = get_embedding(teks_list)  # shape: (6, 384)

# Buat tabel: label, jenis, 5 dimensi pertama + shape info
rows = []
for i, row in df_sample.iterrows():
    vec = embeddings[i]
    rows.append({
        'label': row['label'],
        'jenis': row['jenis'],
        'skor_critic': row['skor_critic'],
        'd0': round(vec[0], 4),
        'd1': round(vec[1], 4),
        'd2': round(vec[2], 4),
        'd3': round(vec[3], 4),
        'd4': round(vec[4], 4),
        '...': '...',
        'd379': round(vec[379], 4),
        'd380': round(vec[380], 4),
        'd381': round(vec[381], 4),
        'd382': round(vec[382], 4),
        'd383': round(vec[383], 4),
    })

df_embed = pd.DataFrame(rows)
print(f"Shape embedding: {embeddings.shape} (6 teks × 384 dimensi)")
print(f"Range nilai: [{embeddings.min():.4f}, {embeddings.max():.4f}]\n")
display(df_embed)

Shape embedding: (6, 384) (6 teks × 384 dimensi)
Range nilai: [-0.5084, 0.5625]



,label,jenis,skor_critic,d0,d1,d2,d3,d4,...,d379,d380,d381,d382,d383
0,good,original,4.250,0.1580,0.1118,-0.1262,0.0603,0.0120,...,-0.0619,0.0593,0.4757,-0.0320,0.1393
1,good,rephrase,4.500,0.2334,0.1041,-0.1489,0.1038,-0.0231,...,-0.0703,0.0704,0.5146,-0.0028,0.1867
2,good,rephrase,4.500,0.1975,0.0812,-0.1304,0.0690,-0.0031,...,-0.0021,0.0951,0.5625,-0.0653,0.0928
3,bad,original,3.375,0.1303,0.0689,0.0420,0.0310,-0.0224,...,-0.1003,0.1068,0.4849,-0.0046,0.0596
4,bad,rephrase,3.500,0.1518,0.0482,0.1108,0.0853,-0.0125,...,-0.1208,0.1314,0.4357,-0.0007,0.0149
5,bad,rephrase,3.500,0.2034,0.1039,0.0087,0.1265,0.0262,...,-0.1082,0.1177,0.5000,0.0002,0.0346


In [18]:
# ============================================================
# HASIL COSINE SIMILARITY — Good 3 & Bad 3
# ============================================================
df_good_cos = df_detailed[df_detailed['label'] == 'good'].head(3)
df_bad_cos = df_detailed[df_detailed['label'] == 'bad'].head(3)
df_cos = pd.concat([df_good_cos, df_bad_cos], ignore_index=True)

display(df_cos[['label', 'jenis', 'skor_critic', 'cosine_similarity']])

,label,jenis,skor_critic,cosine_similarity
0,good,rephrase,4.500,0.888549
1,good,rephrase,4.500,0.887822
2,good,rephrase,4.750,0.838728
3,bad,rephrase,3.500,0.939794
4,bad,rephrase,3.500,0.928217
5,bad,rephrase,3.625,0.928310
